In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score,
    precision_score, recall_score, f1_score
)
import matplotlib.pyplot as plt

In [2]:
dataset_dir = r"C:/Users/RTX2080Ti/Desktop/AD Connectivity/Images/Eyes Closed"
img_height, img_width = 64, 64
batch_size = 32
epochs = 15
seed = 42

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    labels="inferred",
    label_mode="categorical",
    batch_size=batch_size,
    image_size=(img_height, img_width),
    validation_split=0.3,       # 70/30 split
    subset="training",
    seed=seed
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    labels="inferred",
    label_mode="categorical",
    batch_size=batch_size,
    image_size=(img_height, img_width),
    validation_split=0.3,
    subset="validation",
    seed=seed
)

num_classes = len(train_ds.class_names)
class_labels = train_ds.class_names
print("Detected classes:", class_labels)

# Normalize pixel values (0-1)
def normalize_images(image, label):
    return tf.cast(image, tf.float32) / 255.0, label

train_ds = train_ds.map(normalize_images)
test_ds = test_ds.map(normalize_images)

# Convert to arrays (for KFold)
X, y = [], []
for images, labels in train_ds:
    X.extend(images.numpy())
    y.extend(labels.numpy())
X, y = np.array(X), np.array(y)
print("Training data shape:", X.shape, y.shape)

Found 22779 files belonging to 3 classes.
Using 15946 files for training.
Found 22779 files belonging to 3 classes.
Using 6833 files for validation.
Detected classes: ['AD', 'Control', 'FTD']
Training data shape: (15946, 64, 64, 3) (15946, 3)


In [4]:
def build_forcnn_classifier(input_shape=(64, 64, 3), num_classes=3):
    inputs = layers.Input(shape=input_shape)

    def residual_block(x, filters):
        shortcut = x
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.add([x, shortcut])
        x = layers.ReLU()(x)
        return x

    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = residual_block(x, 32)
    x = layers.Conv2D(64, (3, 3), strides=2, padding='same')(x)
    x = layers.ReLU()(x)
    x = residual_block(x, 64)
    x = layers.Conv2D(128, (3, 3), strides=2, padding='same')(x)
    x = layers.ReLU()(x)
    x = residual_block(x, 128)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name="ForCNN_RGB_CV")
    return model

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=seed)
cv_acc, cv_prec, cv_rec, cv_f1 = [], [], [], []

fold = 1
for train_idx, val_idx in kf.split(X):
    print(f"\n===== Fold {fold} =====")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = build_forcnn_classifier((img_height, img_width, 3), num_classes)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=epochs, batch_size=batch_size, verbose=1)

    y_pred = model.predict(X_val)
    y_true_cls = np.argmax(y_val, axis=1)
    y_pred_cls = np.argmax(y_pred, axis=1)

    acc = accuracy_score(y_true_cls, y_pred_cls)
    prec = precision_score(y_true_cls, y_pred_cls, average='macro', zero_division=0)
    rec = recall_score(y_true_cls, y_pred_cls, average='macro', zero_division=0)
    f1 = f1_score(y_true_cls, y_pred_cls, average='macro', zero_division=0)

    cv_acc.append(acc)
    cv_prec.append(prec)
    cv_rec.append(rec)
    cv_f1.append(f1)
    print(f"Fold {fold} → Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")
    fold += 1

print("\n===== 5-Fold Average Results =====")
print(f"Accuracy: {np.mean(cv_acc):.4f}")
print(f"Precision: {np.mean(cv_prec):.4f}")
print(f"Recall: {np.mean(cv_rec):.4f}")
print(f"F1-score: {np.mean(cv_f1):.4f}")


===== Fold 1 =====
Epoch 1/15
399/399 ━━━━━━━━━━━━━━━━━━━━ 104s 249ms/step - accuracy: 0.5370 - loss: 0.9652 - val_accuracy: 0.5859 - val_loss: 0.9666
Epoch 2/15
399/399 ━━━━━━━━━━━━━━━━━━━━ 99s 247ms/step - accuracy: 0.7878 - loss: 0.5305 - val_accuracy: 0.4169 - val_loss: 3.2563
Epoch 3/15
399/399 ━━━━━━━━━━━━━━━━━━━━ 100s 250ms/step - accuracy: 0.8592 - loss: 0.3675 - val_accuracy: 0.8223 - val_loss: 0.4426
Epoch 4/15
399/399 ━━━━━━━━━━━━━━━━━━━━ 103s 258ms/step - accuracy: 0.8942 - loss: 0.2832 - val_accuracy: 0.8721 - val_loss: 0.3415
Epoch 5/15
399/399 ━━━━━━━━━━━━━━━━━━━━ 113s 283ms/step - accuracy: 0.9162 - loss: 0.2312 - val_accuracy: 0.6323 - val_loss: 1.6688
Epoch 6/15
399/399 ━━━━━━━━━━━━━━━━━━━━ 120s 300ms/step - accuracy: 0.9326 - loss: 0.1885 - val_accuracy: 0.8834 - val_loss: 0.2963
Epoch 7/15
399/399 ━━━━━━━━━━━━━━━━━━━━ 116s 290ms/step - accuracy: 0.9403 - loss: 0.1654 - val_accuracy: 0.6959 - val_loss: 0.9958
Epoch 8/15
399/399 ━━━━━━━━━━━━━━━━━━━━ 119s 298ms/step -

In [6]:
print("\n===== Hold-Out Testing on 30% unseen data =====")
final_model = build_forcnn_classifier((img_height, img_width, 3), num_classes)
final_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
final_model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=1)

# Evaluate on hold-out
Y_pred = final_model.predict(test_ds)
y_pred_cls = np.argmax(Y_pred, axis=1)
y_true_cls = np.concatenate([np.argmax(label.numpy(), axis=1) for _, label in test_ds])

print("\nClassification Report (Hold-Out):\n")
print(classification_report(y_true_cls, y_pred_cls, target_names=class_labels))

print("\nConfusion Matrix (Hold-Out):\n")
print(confusion_matrix(y_true_cls, y_pred_cls))


===== Hold-Out Testing on 30% unseen data =====
Epoch 1/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 135s 262ms/step - accuracy: 0.5591 - loss: 0.9030
Epoch 2/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 134s 268ms/step - accuracy: 0.8157 - loss: 0.4684
Epoch 3/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 146s 292ms/step - accuracy: 0.8870 - loss: 0.3015
Epoch 4/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 138s 277ms/step - accuracy: 0.9103 - loss: 0.2374
Epoch 5/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 120s 240ms/step - accuracy: 0.9291 - loss: 0.1920
Epoch 6/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 119s 239ms/step - accuracy: 0.9322 - loss: 0.1768
Epoch 7/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 119s 239ms/step - accuracy: 0.9441 - loss: 0.1480
Epoch 8/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 119s 239ms/step - accuracy: 0.9532 - loss: 0.1281
Epoch 9/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 119s 239ms/step - accuracy: 0.9601 - loss: 0.1119
Epoch 10/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 131s 262ms/step - accuracy: 0.9661 - loss: 0.0963
Epoch 11/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 127s 255m